<p style='text-align: center;'>

## <p style='text-align: center;'> **This is the Notebook for the CIFO Project**

### <p style='text-align: center;'> **Group: C37 Vestigial**




In [1]:
# Numerical arrays and vectorized operations used for triangle genomes and RMSE calculations.
import numpy as np
# Plotting library used to show final images and convergence curves inside the notebook.
import matplotlib.pyplot as plt
# DataFrame structure used to store and analyze results across multiple runs and configurations.
import pandas as pd
# Random utilities used by the genetic algorithm: initialization, selection, and mutation.
from random import randint, choices, sample, random
# OpenCV is used to load images, draw filled triangles, and save generated outputs.
import cv2
# ABC/abstractmethod are used to define a generic optimization solution template.
from abc import ABC, abstractmethod
# deepcopy prevents offspring/elites from accidentally sharing the same object in memory as parents.
from copy import deepcopy
# OS is used for file path manipulations and directory creation.
import os
# Time is used to measure the runtime of the algorithm and to timestamp output files.
import time

## <center> Simulated Annealing

In [2]:
class Solution(ABC):
    """Generic abstract class for an optimization solution."""

    def __init__(self, repr=None):
        # If no representation is provided, create a random one using the subclass method.
        if repr is None:
            repr = self.random_initial_representation()

        # Store the representation/genome of the solution.
        self.repr = repr

    def __repr__(self):
        # Defines how the solution is displayed when printed.
        return str(self.repr)

    @abstractmethod
    def fitness(self):
        # Every specific solution must define how quality/fitness is calculated.
        pass

    @abstractmethod
    def random_initial_representation(self):
        # Every specific solution must define how a random starting solution is created.
        pass

In [3]:
class VermeerSolution(Solution):
    """A candidate painting made from 100 colored triangles."""

    def __init__(self, target_image, repr=None):
        # Store the original image
        self.target_image = target_image

        # Project canvas size
        self.width = 300
        self.height = 400
        self.num_triangles = 100

        super().__init__(repr=repr)

    def random_initial_representation(self):
        random_repr = np.zeros((self.num_triangles, 9), dtype=int)
        for i in range(self.num_triangles):
            cx = randint(0, self.width)
            cy = randint(0, self.height)
            radius = 40
            x1 = np.clip(cx + randint(-radius, radius), 0, self.width)
            y1 = np.clip(cy + randint(-radius, radius), 0, self.height)
            x2 = np.clip(cx + randint(-radius, radius), 0, self.width)
            y2 = np.clip(cy + randint(-radius, radius), 0, self.height)
            x3 = np.clip(cx + randint(-radius, radius), 0, self.width)
            y3 = np.clip(cy + randint(-radius, radius), 0, self.height)
            r, g, b = [randint(0, 255) for _ in range(3)]
            random_repr[i] = [x1, y1, x2, y2, x3, y3, r, g, b]
        return random_repr

    def render_canvas(self):
        canvas = np.zeros((self.height, self.width, 3), dtype=np.uint8)
        for gene in self.repr:
            pts = np.array([[gene[0], gene[1]], [gene[2], gene[3]], [gene[4], gene[5]]], np.int32)
            pts = pts.reshape((-1, 1, 2))
            color = (int(gene[6]), int(gene[7]), int(gene[8]))
            cv2.fillPoly(canvas, [pts], color)
        return canvas

    def fitness(self):
        """Calculates RMSE + Geometric Penalty (Shoelace Formula)"""
        generated_image = self.render_canvas()
        target_float = self.target_image.astype(np.float32)
        gen_float = generated_image.astype(np.float32)

        # 1. Erro Visual (RMSE)
        diff = target_float - gen_float
        sq_diff = np.square(diff)
        mse = np.mean(sq_diff)
        rmse = np.sqrt(mse)

        return rmse

    def get_random_neighbor(self):
        """
        Generates a new solution optimized for Simulated Annealing.
        Uses Gaussian distributions and Heuristic color stealing.
        """
        neighbor = VermeerSolution(
            target_image=self.target_image, 
            repr=self.repr.copy()
        )
        
        pm = 0.03 
        
        for i in range(neighbor.num_triangles):
            if random() < pm:
                mutation_type = random()
                
                if mutation_type < 0.90:
                    # 90% of mutations: Nudge
                    neighbor.repr[i][0:6] += np.random.randint(-10, 11, 6)  # Coordinates
                    neighbor.repr[i][6:9] += np.random.randint(-15, 16, 3)  # Colors
                    
                    # Secure bounds after mutation
                    neighbor.repr[i][0::2] = np.clip(neighbor.repr[i][0::2], 0, neighbor.width)
                    neighbor.repr[i][1::2] = np.clip(neighbor.repr[i][1::2], 0, neighbor.height)
                    neighbor.repr[i][6:9] = np.clip(neighbor.repr[i][6:9], 0, 255)
                    
                else:
                    # 10% of mutations: Random Reset
                    neighbor.repr[i] = [
                        randint(0, neighbor.width),
                        randint(0, neighbor.height),
                        randint(0, neighbor.width),
                        randint(0, neighbor.height),
                        randint(0, neighbor.width),
                        randint(0, neighbor.height),
                        randint(0, 255),  # Random red
                        randint(0, 255),  # Random green
                        randint(0, 255)   # Random blue
                    ]
                    
        return neighbor

In [4]:
def simulated_annealing(
    initial_solution: Solution,
    C: float,
    L: int,
    cooling_rate: float,
    max_iter: int = 10,
    verbose: bool = False,
):
    """
    Generic Simulated Annealing implementation kept from class material.

    NOTE: VermeerSolution would need get_random_neighbor() implemented for this to work on the painting problem.
    """
    # Start from the provided initial solution and calculate its fitness.
    current_solution = initial_solution
    current_fitness = current_solution.fitness()

    # Best global solution as SA can accept worse ones
    best_solution = initial_solution
    best_fitness = current_fitness

    C = C
    rmse_history = []  
    total_evals = 0

    for iteration in range(1, max_iter + 1):

        for _ in range(L):
            # Generate a neighbor with a slight mutation
            neighbor = current_solution.get_random_neighbor()
            neighbor_fitness = neighbor.fitness()
            total_evals += 1

            delta = neighbor_fitness - current_fitness   # less than 0, improvement

            if delta <= 0:
                # if the neighbor is better, accept it as current solution
                current_solution = neighbor
                current_fitness = neighbor_fitness
            else:
                # if the neighbor is worse, accept with probability of Boltzmann
                prob = np.exp(-delta / C) if C > 1e-10 else 0.0
                if random() < prob:
                    current_solution = neighbor
                    current_fitness = neighbor_fitness

            # Update global best
            if current_fitness < best_fitness:
                best_fitness = current_fitness
                # Save a copy of the best solution
                best_solution = VermeerSolution(
                    target_image=current_solution.target_image,
                    repr=current_solution.repr.copy(),
                )

            rmse_history.append(current_fitness)

        # Geometric cooling
        C *= cooling_rate

        if verbose and iteration % 50 == 0:
            print(
                f"  Iter {iteration:5d}/{max_iter} | "
                f"C={C:.4f} | "
                f"RMSE atual={current_fitness:.2f} | "
                f"Best={best_fitness:.2f}"
            )

    return best_solution, rmse_history


In [7]:
# Path to local copies of the target image.
joao_path = r"C:\Users\joaoa\Desktop\CIFO\data\girl_pearl_earing.png"
goncalo_path = r"C:\CIfO\Project\data\girl_pearl_earing.png"
rafa_path = r"Project\girl_pearl_earing.png"
gui_path = r"C:\Users\User\Desktop\semestre_2\Inteligencia_Computacional_Otimizacao\CIFO_Group37_Vestigial\Project\girl_pearl_earing.png"
gui_path_2 = r"C:\Users\User\Documents\GitHub\CIFO_Group37_Vestigial\Project\girl_pearl_earing.png"

# Load the original image using OpenCV.
original_image = cv2.imread(gui_path_2)

# Fail early with a clear error if the path is wrong or the image cannot be read.
if original_image is None:
    raise FileNotFoundError(f"OpenCV could not find the image at: {gui_path_2}")

In [ ]:
import os
import time
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import copy

# --- 1. CONFIGURAÇÃO DO LOTE ---
EXP_NAME = "Simulated_Annealing"
NOTES = "SA Optimization with 300.000 evaluations."
NUM_RUNS = 20

# Parâmetros Otimizados do SA
L_PARAM = 150         
MAX_ITER_PARAM = 2000 
C_INICIAL = 10.0      
COOLING_RATE_PARAM = 0.996

print(f"\n[{time.strftime('%H:%M:%S')}] Starting SA batch: {EXP_NAME} ({NUM_RUNS} Runs)")

variant_dir = os.path.join("data", "results", EXP_NAME)
if not os.path.exists(variant_dir):
    os.makedirs(variant_dir)

all_runs_history = []
final_rmses = []

start_time = time.time()

# --- 2. CICLO SEQUENCIAL (Uma Run de cada vez) ---
for run in range(1, NUM_RUNS + 1):
    print(f"\nStarting SA Run {run}/{NUM_RUNS}...")
    
    run_dir = os.path.join(variant_dir, f"Run_{run}")
    if not os.path.exists(run_dir):
        os.makedirs(run_dir)
        
    # ⚠️ ALERTA DE JUSTIÇA ACADÉMICA (PONTO 4):
    # Neste momento, isto gera uma solução aleatória cega.
    # Quando fores comparar com o GA no final, substitui a linha abaixo por:
    # sa_initial = copy.deepcopy(melhor_quadro_inicial_do_ga)
    sa_initial = VermeerSolution(target_image=original_image)

    # Executar o SA (verbose=False para não encher o ecrã)
    best_sa_sol, sa_history_raw = simulated_annealing(
        initial_solution=sa_initial,
        C=C_INICIAL, 
        L=L_PARAM, 
        cooling_rate=COOLING_RATE_PARAM, 
        max_iter=MAX_ITER_PARAM, 
        verbose=True
    )
    
    # Processar o histórico para "Best-So-Far" (para o gráfico não andar aos saltos)
    best_so_far_sa = []
    current_best = sa_history_raw[0]
    for score in sa_history_raw:
        if score < current_best:
            current_best = score
        best_so_far_sa.append(current_best)
        
    final_rmse = best_so_far_sa[-1]
    print(f"SA Run {run} completed. Final RMSE: {final_rmse:.2f}")

    # Guardar Hiperparâmetros e Histórico
    hyperparameter_path = os.path.join(run_dir, "hyperparameters.txt")
    with open(hyperparameter_path, "w") as file:
        file.write(f"Simulated Annealing Hyperparameters | Run Number: {run}\n\n")
        if NOTES: file.write(f"Notes: {NOTES}\n\n")
        file.write(f"Initial Temperature (C): {C_INICIAL}\n")
        file.write(f"Cooling Rate: {COOLING_RATE_PARAM}\n")
        file.write(f"Iterations at each Temp (L): {L_PARAM}\n")
        file.write(f"Max Temperature Steps (max_iter): {MAX_ITER_PARAM}\n")
        file.write(f"Total Evaluations: {L_PARAM * MAX_ITER_PARAM}\n\n")
        
        # Guardamos de 100 em 100 iterações para o TXT não ficar gigantesco
        file.write(f"--- ITERATION HISTORY ---\n")
        step = (L_PARAM * MAX_ITER_PARAM) // 40 
        
        for i in range(0, len(best_so_far_sa), step):
            # Dividir por L_PARAM converte o número da avaliação para a "Iteração" equivalente (0, 50, 100...)
            equiv_iter = i // L_PARAM
            file.write(f"Iter {equiv_iter} | Fitness: {best_so_far_sa[i]:.2f}\n")
        file.write(f"\n--- FINAL RESULTS ---\nFinal RMSE Score: {final_rmse:.2f}\n")

    # Guardar Imagem Final
    cv2.imwrite(os.path.join(run_dir, "final_result.png"), best_sa_sol.render_canvas())
    
    # Guardar Gráfico de Convergência (Individual)
    plt.figure(figsize=(8, 4))
    plt.plot(best_so_far_sa, color='orange', linewidth=1.5)
    plt.title(f"SA Convergence - Run {run} (Final RMSE: {final_rmse:.2f})")
    plt.xlabel("Temperature Steps (max_iter)")
    plt.ylabel("RMSE (Best-so-Far)")
    plt.grid(True)
    plt.savefig(os.path.join(run_dir, "convergence_plot.png"), bbox_inches='tight')
    plt.close()
    
    # Adicionar dados globais para os CSVs
    all_runs_history.append(best_so_far_sa)
    final_rmses.append(final_rmse)

history = [run_history[::L_PARAM] for run_history in all_runs_history]

# Garantir que ficam todos do mesmo tamanho antes de converter para numpy
min_len = min(len(r) for r in history)
aligned_history = [r[:min_len] for r in history]

history_matrix = np.array(aligned_history)

print(f"Best Mean RMSE: {np.mean(history_matrix, axis=0)[-1]:.2f}")

# Statistical Summary CSV
stats_df = pd.DataFrame({
    "Iteration": np.arange(min_len),
    "Mean_RMSE": np.mean(history_matrix, axis=0),
    "Std_Dev": np.std(history_matrix, axis=0),
    "Best_RMSE": np.min(history_matrix, axis=0)
})
stats_df.to_csv(os.path.join(variant_dir, "statistical_summary.csv"), index=False)


[11:44:44] Starting SA batch: Simulated_Annealing (30 Runs)

Starting SA Run 1/30...
  Iter    50/2000 | C=8.1840 | RMSE atual=90.91 | Best=81.55
  Iter   100/2000 | C=6.6978 | RMSE atual=91.04 | Best=78.95
  Iter   150/2000 | C=5.4815 | RMSE atual=87.90 | Best=78.95
  Iter   200/2000 | C=4.4861 | RMSE atual=93.73 | Best=78.95
  Iter   250/2000 | C=3.6714 | RMSE atual=85.25 | Best=76.76
  Iter   300/2000 | C=3.0047 | RMSE atual=90.67 | Best=76.76
  Iter   350/2000 | C=2.4591 | RMSE atual=78.15 | Best=76.76
  Iter   400/2000 | C=2.0125 | RMSE atual=80.93 | Best=76.76
  Iter   450/2000 | C=1.6470 | RMSE atual=75.50 | Best=73.64
  Iter   500/2000 | C=1.3479 | RMSE atual=85.33 | Best=69.52
  Iter   550/2000 | C=1.1032 | RMSE atual=68.45 | Best=68.03
  Iter   600/2000 | C=0.9028 | RMSE atual=74.22 | Best=64.77
  Iter   650/2000 | C=0.7389 | RMSE atual=68.68 | Best=60.24
  Iter   700/2000 | C=0.6047 | RMSE atual=64.84 | Best=56.62
  Iter   750/2000 | C=0.4949 | RMSE atual=62.72 | Best=56.62

ValueError: Length mismatch: Expected axis has 300000 elements, new values have 2000 elements